In [1]:
import os
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

current_dir = os.getcwd()
dataset_dir = os.path.abspath(os.path.join(current_dir, "..", "dataset"))
test_path = os.path.join(dataset_dir, "test.csv")
model_save_dir = os.path.abspath(os.path.join(current_dir, "..", "models", "deberta-v3-detector"))

print("Loading test data...")
test_df = pd.read_csv(test_path).dropna(subset=["text", "label"])
test_df["text"] = test_df["text"].astype(str)
test_df["label"] = test_df["label"].astype(int)
print(f"Loaded {len(test_df)} test samples successfully!")


d:\Deceptiscan\DECEPTISCAN---BROWSER-EXTENSION-main\Deceptiscan\extension\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading test data...
Loaded 14332 test samples successfully!


In [3]:
import torch      # <-- Add this line
print(f"Loading local fine-tuned model and tokenizer from {model_save_dir}...")
tokenizer = AutoTokenizer.from_pretrained(model_save_dir)

# Bypasses the PyTorch 2.5/2.6 security check for local files
model = AutoModelForSequenceClassification.from_pretrained(model_save_dir, weights_only=False)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()
print(f"Model successfully loaded on device: {device}")



Loading local fine-tuned model and tokenizer from d:\Deceptiscan\DECEPTISCAN---BROWSER-EXTENSION-main\Deceptiscan\extension\backend\models\deberta-v3-detector...


The tokenizer you are loading from 'd:\Deceptiscan\DECEPTISCAN---BROWSER-EXTENSION-main\Deceptiscan\extension\backend\models\deberta-v3-detector' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Model successfully loaded on device: cuda


In [4]:
import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix

texts = test_df["text"].tolist()
labels = test_df["label"].tolist()

print(f"Running predictions on {len(texts)} test samples...")
predictions = []
probabilities = []

# Batch predictions to optimize memory
batch_size = 16
with torch.no_grad():
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)
        
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)
        
        predictions.extend(preds)
        probabilities.extend(probs[:, 1])

# 1. Print Classification Report
print("\n" + "="*40)
print("CLASSIFICATION REPORT")
print("="*40)
print(classification_report(labels, predictions, target_names=["Genuine (0)", "Fake (1)"]))

# 2. Print Confusion Matrix
print("\n" + "="*40)
print("CONFUSION MATRIX")
print("="*40)
cm = confusion_matrix(labels, predictions)
print(f"True Negatives (Genuine identified as Genuine): {cm[0][0]}")
print(f"False Positives (Genuine identified as Fake):    {cm[0][1]}")
print(f"False Negatives (Fake identified as Genuine):    {cm[1][0]}")
print(f"True Positives (Fake identified as Fake):       {cm[1][1]}")
print("="*40)

# 3. View Sample Errors (optional diagnostics)
error_indices = [idx for idx, (l, p) in enumerate(zip(labels, predictions)) if l != p]
if error_indices:
    print("\n" + "="*40)
    print("SAMPLE MISCLASSIFICATIONS")
    print("="*40)
    for idx in error_indices[:5]:
        print(f"Index: {idx}")
        print(f"Actual: {labels[idx]} | Predicted: {predictions[idx]} (Prob of Fake: {probabilities[idx]:.4f})")
        print(f"Text: {texts[idx][:150]}...")
        print("-"*40)


Running predictions on 14332 test samples...

CLASSIFICATION REPORT
              precision    recall  f1-score   support

 Genuine (0)       0.89      0.77      0.83      5916
    Fake (1)       0.85      0.93      0.89      8416

    accuracy                           0.87     14332
   macro avg       0.87      0.85      0.86     14332
weighted avg       0.87      0.87      0.87     14332


CONFUSION MATRIX
True Negatives (Genuine identified as Genuine): 4566
False Positives (Genuine identified as Fake):    1350
False Negatives (Fake identified as Genuine):    551
True Positives (Fake identified as Fake):       7865

SAMPLE MISCLASSIFICATIONS
Index: 8
Actual: 0 | Predicted: 1 (Prob of Fake: 0.9025)
Text: interesting design and i like the metal cabinet the heater is nice for heating a select area as the warmth is directed towards the area it is directed...
----------------------------------------
Index: 12
Actual: 1 | Predicted: 0 (Prob of Fake: 0.0047)
Text: if you re like me and fou

In [9]:
import torch
import numpy as np

def predict_review(review_text):
    # 1. Tokenize the input text
    inputs = tokenizer(
        review_text,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)
    
    # 2. Disable gradient tracking and get predictions
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Calculate probabilities for both classes (0: Genuine, 1: Fake)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
        
    # 3. Determine the class and confidence score
    pred_class_id = np.argmax(probs)
    confidence = probs[pred_class_id] * 100
    
    class_label = "Fake Review (Suspicious)" if pred_class_id == 1 else "Genuine Review (Trustworthy)"
    
    # Print formatted output
    print(f"Review: \"{review_text[:120]}...\"")
    print(f"Prediction: {class_label}")
    print(f"Confidence: {confidence:.2f}%")
    print(f"Genuine Probability: {probs[0]*100:.2f}% | Fake Probability: {probs[1]*100:.2f}%")
    print("-" * 50)

# --- Test it on your own sentences here! ---

predict_review("Customers find the plush toy soft and huggable, with one mentioning its smooth texture, and many noting it's cute and perfect for 3-year-olds. The toy receives positive feedback for its size, being just right for little hands, and customers appreciate its spot-on colors and good quality. Customers consider it a great gift for toddlers and find it worth the price tag.")
predict_review("This product is absolute garbage! It broke on the very first day. Do not buy!")
predict_review("AMAZING product! BEST EVER! MUST BUY! I love this so much, perfect product, 100% recommended!!!!!!!!")
predict_review("I bought this table last month. It does the job well, has minor scratches on the legs, but overall fits my dining room setup nicely.")


Review: "Customers find the plush toy soft and huggable, with one mentioning its smooth texture, and many noting it's cute and pe..."
Prediction: Fake Review (Suspicious)
Confidence: 64.60%
Genuine Probability: 35.40% | Fake Probability: 64.60%
--------------------------------------------------
Review: "This product is absolute garbage! It broke on the very first day. Do not buy!..."
Prediction: Fake Review (Suspicious)
Confidence: 92.51%
Genuine Probability: 7.49% | Fake Probability: 92.51%
--------------------------------------------------
Review: "AMAZING product! BEST EVER! MUST BUY! I love this so much, perfect product, 100% recommended!!!!!!!!..."
Prediction: Fake Review (Suspicious)
Confidence: 99.72%
Genuine Probability: 0.28% | Fake Probability: 99.72%
--------------------------------------------------
Review: "I bought this table last month. It does the job well, has minor scratches on the legs, but overall fits my dining room s..."
Prediction: Genuine Review (Trustworthy)
Co